<a href="https://colab.research.google.com/github/anandourado/Raspagem-da-Folha-de-S.-Paulo/blob/main/Raspagem_da_Folha_de_S_Paulo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Instalação**

In [10]:
!pip install -q git+https://github.com/bdcdo/raspe.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


# **Raspagem de dados**

In [11]:
import logging
logging.getLogger("FOLHA").setLevel(logging.INFO)

import raspe

folha = raspe.folha()

dados = folha.raspar(
    pesquisa=["bolsonaro", "lula", "eleições", "eleitoral"],
    site="todos",
    data_inicio="2023-01-1",
    data_fim="2023-01-08",
)

print(f"Total bruto: {len(dados)}")

# remover duplicatas (notícias que mencionam os dois nomes)
termo_agrupado = dados.groupby("link")["termo_busca"].apply(lambda x: ", ".join(sorted(set(x))))
dados = dados.drop_duplicates(subset="link").reset_index(drop=True)
dados = dados.merge(termo_agrupado.rename("termos_busca"), on="link")
dados = dados.drop(columns=["termo_busca"])

print(f"Total após remover duplicatas: {len(dados)}")
dados.head()

ValidationError: 'data_inicio' está em formato inválido: '2023-01-1'. Use um dos formatos: YYYY-MM-DD, DD/MM/YYYY ou YYYYMMDD.

# **Filtrar apenas Opinião**

In [ ]:
dados_opiniao = dados[dados["link"].str.contains("/opiniao/", na=False)].reset_index(drop=True)
print(f"Notícias de Opinião: {len(dados_opiniao)}")
dados_opiniao.head()

# **Diagnóstico do HTML antes de rodar tudo**

In [ ]:
import requests
from bs4 import BeautifulSoup
import json

url_teste = dados_opiniao["link"].iloc[0]
print("URL testada:", url_teste)

headers = {"User-Agent": "Mozilla/5.0"}
r = requests.get(url_teste, headers=headers, timeout=10)
soup = BeautifulSoup(r.text, "html.parser")

print("\n--- Meta tags relevantes ---")
for m in soup.find_all("meta"):
    nome = m.get("name") or m.get("property")
    if nome and ("author" in nome.lower() or "byline" in nome.lower() or "writer" in nome.lower()):
        print(nome, "->", m.get("content"))

print("\n--- JSON-LD encontrado ---")
for script in soup.find_all("script", type="application/ld+json"):
    try:
        data = json.loads(script.string)
        print(json.dumps(data, indent=2, ensure_ascii=False)[:1000])
        print("---")
    except (json.JSONDecodeError, TypeError):
        continue

print("\n--- Elementos com 'autor'/'author' na classe ---")
for tag in soup.find_all(class_=True):
    classes = " ".join(tag.get("class", []))
    if "autor" in classes.lower() or "author" in classes.lower():
        print(tag.name, classes, "->", tag.get_text(strip=True)[:100])

print("\n--- Links para /colunistas/ ou /colunas/ ---")
for a in soup.find_all("a", href=True):
    if "/colunistas/" in a["href"] or "/colunas/" in a["href"]:
        print(a["href"], "->", a.get_text(strip=True))

# **Coleta de autor e texto**




In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import time

GENERICOS = {"folha.uol.com.br", "folha de s.paulo", "folha de sao paulo", "uol", "folhapress"}

def pegar_autor_e_texto(url, timeout=15):
    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        r = requests.get(url, headers=headers, timeout=timeout)
        r.encoding = "utf-8"  # corrige o problema de acentuação
        soup = BeautifulSoup(r.text, "html.parser")

        # --- autor ---
        autor = None
        candidatos = []
        for script in soup.find_all("script", type="application/ld+json"):
            try:
                data = json.loads(script.string)
                if isinstance(data, dict) and "author" in data:
                    a = data["author"]
                    if isinstance(a, dict) and a.get("name"):
                        candidatos.append(a["name"].strip())
                    elif isinstance(a, list):
                        for item in a:
                            if isinstance(item, dict) and item.get("name"):
                                candidatos.append(item["name"].strip())
            except (json.JSONDecodeError, TypeError):
                continue

        meta = soup.find("meta", attrs={"name": "author"})
        if meta and meta.get("content"):
            candidatos.append(meta["content"].strip())

        for c in candidatos:
            if c.lower() not in GENERICOS:
                autor = c
                break

        # --- texto ---
        texto = None
        corpo = soup.find("div", class_="c-news__body")
        if corpo:
            texto = corpo.get_text(" ", strip=True)

        return autor, texto

    except requests.RequestException:
        return None, None


autores = []
textos = []
total = len(dados_opiniao)

for i, link in enumerate(dados_opiniao["link"], start=1):
    autor, texto = pegar_autor_e_texto(link)
    autores.append(autor)
    textos.append(texto)
    if i % 10 == 0 or i == total:
        print(f"{i}/{total} processadas")
    time.sleep(1)

dados_opiniao["autor"] = autores
dados_opiniao["texto"] = textos
dados_opiniao.head()

# **Remover linhas sem autor (editoriais) e checar texto**

In [ ]:
antes = len(dados_opiniao)
dados_opiniao = dados_opiniao[dados_opiniao["autor"].notna()].reset_index(drop=True)
depois = len(dados_opiniao)
print(f"Removidos {antes - depois} editoriais sem autor. Restaram {depois} notícias assinadas.")
dados_opiniao.head()

# **Salvar**

In [ ]:
dados_opiniao.to_excel("folha_opiniao.xlsx", index=False)
from google.colab import files
files.download("folha_opiniao.xlsx")